"""
题目：魔法学校智力测试 (网格移动路径数)
核心思路：
1. 破局点：行和列的移动规则严格独立，可以将二维网格降维成两个一维的“爬楼梯”问题。
2. 数学化：行方案数 * 列方案数 * 组合数 C(行步数+列步数, 行步数)。
3. 极致优化：使用“前缀乘积 + 费马小定理求逆元”实现 O(1) 求解区间连乘，预处理阶乘实现 O(1) 求组合数。
"""

In [2]:
from typing import List, Tuple

MOD = 1_000_000_007
MAX_N = 200_005


class Solution:
    # 类级别预处理阶乘/逆阶乘，共享于所有实例
    _fact = [1] * MAX_N
    _inv_fact = [1] * MAX_N
    _precomputed = False

    @classmethod
    def _precompute(cls):
        if cls._precomputed:
            return
        for i in range(1, MAX_N):
            cls._fact[i] = cls._fact[i - 1] * i % MOD
        cls._inv_fact[MAX_N - 1] = pow(cls._fact[MAX_N - 1], MOD - 2, MOD)
        for i in range(MAX_N - 2, -1, -1):
            cls._inv_fact[i] = cls._inv_fact[i + 1] * (i + 1) % MOD
        cls._precomputed = True

    def _nCr(self, n: int, r: int) -> int:
        if r < 0 or r > n:
            return 0
        return self._fact[n] * self._inv_fact[r] % MOD * self._inv_fact[n - r] % MOD

    def _build_dim(self, arr: List[int]) -> Tuple[dict, List[int]]:
        """将一维数组压缩为等级映射 + 前缀乘积表。"""
        unique = sorted(set(arr))
        rank = {v: i for i, v in enumerate(unique)}
        counts = [0] * len(unique)
        for v in arr:
            counts[rank[v]] += 1
        # prefix_prod[i] = counts[0] * ... * counts[i-1]
        prefix_prod = [1] * (len(unique) + 1)
        for i, c in enumerate(counts):
            prefix_prod[i + 1] = prefix_prod[i] * c % MOD
        return rank, prefix_prod

    def _get_ways(self, sv: int, tv: int, rank: dict, pref: List[int]) -> Tuple[int, int]:
        """返回 (该维度走法数, 步数)。"""
        if sv == tv:
            return 1, 0
        if sv > tv or sv not in rank or tv not in rank:
            return 0, 0
        rs, rt = rank[sv], rank[tv]
        steps = rt - rs
        if steps == 1:
            return 1, 1
        # 区间连乘：pref[rt] / pref[rs+1]（模意义除法）
        ways = pref[rt] * pow(pref[rs + 1], MOD - 2, MOD) % MOD
        return ways, steps

    def solve(self, n: int, m: int, R: List[int], C: List[int],
              queries: List[Tuple[int, int, int, int]]) -> List[int]:
        self._precompute()
        rank_R, pref_R = self._build_dim(R)
        rank_C, pref_C = self._build_dim(C)

        results = []
        for sr, sc, tr, tc in queries:
            # 校验单调性
            if (sr != tr and R[sr] >= R[tr]) or (sc != tc and C[sc] >= C[tc]):
                results.append(0)
                continue

            ways_r, step_r = self._get_ways(R[sr], R[tr], rank_R, pref_R)
            ways_c, step_c = self._get_ways(C[sc], C[tc], rank_C, pref_C)

            if ways_r == 0 or ways_c == 0:
                results.append(0)
                continue

            ans = ways_r * ways_c % MOD * self._nCr(step_r + step_c, step_r) % MOD
            results.append(ans)

        return results

In [ ]:
# --- 测试 ---
sol = Solution()

# R = [1, 2, 3]（三行权重严格递增，每个等级各1行）
# C = [1, 2, 3]（三列权重严格递增，每个等级各1列）
R = [1, 2, 3]
C = [1, 2, 3]
queries = [
    (0, 0, 2, 2),   # 跨2行2列
    (0, 0, 1, 1),   # 跨1行1列
    (0, 0, 0, 0),   # 原点到原点
    (2, 2, 0, 0),   # 逆序，不可达
]
print(sol.solve(3, 3, R, C, queries))
# 期望：[6, 2, 1, 0]
# (0,0)->(2,2): step_r=2, step_c=2, ways_r=1, ways_c=1 → 1*1*C(4,2) = 6
#   6条路径：RRCC / RCRC / RCCR / CRRC / CRCR / CCRR
# (0,0)->(1,1): step_r=1, step_c=1 → 1*1*C(2,1) = 2
# (0,0)->(0,0): 原地不动 → 1
# (2,2)->(0,0): R[2]=3 > R[0]=1，能量下降 → 0